# 01 Problem definition and data

*This notebook covers **Step 1 (Problem definition)** and **Step 2 (Data)** of the project.*

## 1. Problem definition

**Is the DLS method fair in T20 cricket?** DLS was designed for 50-over matches and later applied to T20. If its par scores are fair, a chasing team sitting exactly on par should win about 50% of the time.

As an ML problem: given the state of a chase (runs needed, balls left, wickets left), predict the probability the chasing team wins. That makes it **binary classification**, but the output that matters is the probability, not the yes/no, because the model is then used as a tool to test DLS.

## 2. Data

[Cricsheet](https://cricsheet.org/downloads/) ball-by-ball data for men's T20 internationals and IPL, JSON format. Structured data: one file per match. Downloaded 4 November 2026 and used under the ODC-By licence. The raw files aren't in this repo (394MB), but the processed output, `data/chases.csv`, is. To rebuild it, download both zips and unzip into `data/t20s_male_json/` and `data/ipl_male_json/`.

Cricsheet withholds matches involving the Afghanistan men's team or played in the Afghanistan Premier League (166 T20Is at the time of download), so they aren't included.

This notebook turns it into one table: every delivery of every second innings, with the match state at that point and whether the chasing team went on to win.

## Looking at one match

Each match is one JSON file: `info` holds metadata and the result, `innings` holds every delivery. The second innings carries a `target` (runs and overs), which is reduced in rain-affected matches.

This file happens to be a tie decided by a super over, which is one reason ties are dropped below.

In [3]:
import json, glob

files = glob.glob("data/t20s_male_json/*.json")
print(len(files))

with open(files[0]) as f:
    m = json.load(f)

print(m["info"].keys())
print(m["info"]["outcome"])
print(m["innings"][1]["team"], m["innings"][1].get("target"))
print(m["innings"][1]["overs"][0]["deliveries"][0])

3649
dict_keys(['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'match_type_number', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue'])
{'result': 'tie', 'eliminator': 'India'}
Sri Lanka {'overs': 20, 'runs': 138}
{'actual_delivery': '0.1', 'batter': 'P Nissanka', 'bowler': 'KK Ahmed', 'non_striker': 'BKG Mendis', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}


## Parser

One row per second-innings delivery. Decisions worth noting:

- **Ties and no-results are dropped.** Their outcome can't be labelled cleanly as a chasing win or loss. It's a small share of matches.
- **Wides and no-balls don't use up a ball.** They add runs but are re-bowled, so they don't reduce `balls_left`.
- **Reduced overs use cricket notation.** A target over 13.5 overs means 13 overs and 5 balls (83 balls), not 13.5 × 6 = 81. Treating it as a decimal gave negative `balls_left` in 27 rain-affected matches, the exact matches this project tests. Converted properly below.
- **A few matches had a 7-ball over** (umpire miscount), giving `balls_left` of -1. Clipped to 0.
- **DLS matches are flagged, not removed.** The model is trained on normal chases and then used to test the DLS matches, so training on them would be circular.

In [4]:
import pandas as pd

def parse_match(path, comp):
    with open(path) as f:
        m = json.load(f)
    info = m["info"]
    out = info["outcome"]

    if "winner" not in out or len(m["innings"]) < 2:
        return []

    inn1, inn2 = m["innings"][0], m["innings"][1]
    target = inn2.get("target", {})
    target_runs = target.get("runs")
    target_overs = target.get("overs", info.get("overs", 20))
    if target_runs is None:
        target_runs = sum(d["runs"]["total"] for o in inn1["overs"] for d in o["deliveries"]) + 1

    chasing = inn2["team"]
    won = int(out["winner"] == chasing)
    dls = "method" in out
    total_balls = int(target_overs) * 6 + round((target_overs % 1) * 10)

    rows, runs, wkts, legal = [], 0, 0, 0
    for o in inn2["overs"]:
        for d in o["deliveries"]:
            runs += d["runs"]["total"]
            wkts += len(d.get("wickets", []))
            ex = d.get("extras", {})
            if "wides" not in ex and "noballs" not in ex:
                legal += 1
            rows.append({
                "match_id": path.split("/")[-1][:-5],
                "comp": comp,
                "date": info["dates"][0],
                "target": target_runs,
                "runs": runs,
                "wickets": wkts,
                "balls_left": max(total_balls - legal, 0),
                "runs_needed": target_runs - runs,
                "dls": dls,
                "won": won,
            })
    return rows

all_rows = []
for comp, folder in [("t20i", "t20s_male_json"), ("ipl", "ipl_male_json")]:
    for p in glob.glob(f"data/{folder}/*.json"):
        all_rows += parse_match(p, comp)

df = pd.DataFrame(all_rows)
print(df.shape)
df.head()

(514444, 10)


,match_id,comp,date,target,runs,wickets,balls_left,runs_needed,dls,won
0,1339605,t20i,2023-03-26,259,4,0,119,255,False,1
1,1339605,t20i,2023-03-26,259,8,0,118,251,False,1
2,1339605,t20i,2023-03-26,259,8,0,117,251,False,1
3,1339605,t20i,2023-03-26,259,8,0,116,251,False,1
4,1339605,t20i,2023-03-26,259,9,0,115,250,False,1


## Sanity checks

- Each match has exactly one result
- Chasing teams win 50.8% of T20Is and 54.7% of IPL matches. Chasing has a known edge in the IPL (dew, flat pitches), so competition is kept as a feature
- 149 DLS-affected matches set aside for the final test
- `balls_left` runs from 0 to 120 with no negatives

In [5]:
print(df.groupby("comp")["match_id"].nunique())
print(df.groupby("comp").apply(lambda g: g.drop_duplicates("match_id")["won"].mean()))
print(df.drop_duplicates("match_id")["dls"].sum(), "DLS matches")
print(df["balls_left"].min(), df["balls_left"].max())
df.to_csv("data/chases.csv", index=False)

comp
ipl     1218
t20i    3529
Name: match_id, dtype: int64
comp
ipl     0.546798
t20i    0.507793
dtype: float64
149 DLS matches
0 120
